# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heyzara124-hub/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding — "What Predicts Health?" (ML Appendix, Random Forest on health score).**
The paper trains a Random Forest to find which features predict `health_score`, and reports
Average Position, Impressions, and Scroll Depth as the top three by importance. The paper
itself flags the issue: `health_score` is a composite built directly from position,
impressions, CTR, and scroll depth — the same fields the model is "predicting" from.

*My methodology question:* holdout testing checks whether a model overfits noise, but it
can't fix a target that is mathematically constructed from its own inputs. No matter how
honest the split is, a feature that is literally one-fifth of the label's formula will always
look important. So the headline number here (43% importance on Average Position) tells us the
model rediscovered the scoring formula, not that position predicts something new. The paper
does note this in a caveat, which is the right call — I'd just make the caveat louder, since
the chart alone (without reading the fine print) reads as a discovery.

**Finding — "The Anatomy of Growing Content" (Finding #1, direct comparison).**
The paper compares pages with rising impressions against pages with falling impressions, and
reports growing pages are about 38% longer and 20% younger on average. The label
(`trend_direction`, growing vs. declining) comes from an observed 30-day-vs-prior-30-day
comparison, not a defined rule, so it passes the "target must be observed" bar.

*My methodology question:* this is an observational comparison, not an experiment, so the
direction of cause and effect is still open. The paper's own recommendation ("expand thin
pages") assumes length drives growth. But an equally consistent story is reverse causation:
editors notice which pages are already gaining traction and invest more words into the
winners, so growth causes length rather than the other way round. The paper is careful to call
this "directionally robust" rather than causal, which is the right hedge — the open question is
just which direction the arrow points.

In [1]:
# No computation needed for this section — both findings are read directly from
# docs/flyrank-seo-research-march-2026.pdf (Finding #1, page 6, and the ML Appendix, page 27).
print('Findings reviewed: (1) Random Forest -> health_score, ML Appendix page 27')
print('                    (2) Growing vs declining content profile, Finding #1 page 6')

Findings reviewed: (1) Random Forest -> health_score, ML Appendix page 27
                    (2) Growing vs declining content profile, Finding #1 page 6


*(Rebuilding the same data and features from Week 5 here, so this notebook runs standalone.)*

In [2]:
import pandas as pd
import numpy as np
import os

if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.exists('flyrank-ml-internship'):
        !git clone -q https://github.com/heyzara124-hub/flyrank-ml-internship.git
    os.chdir('flyrank-ml-internship')

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)].copy()
tier_median_ctr = visible.groupby('position_tier')['ctr'].median()
visible['tier_median_ctr'] = visible['position_tier'].map(tier_median_ctr)
visible['ctr_gap'] = visible['ctr'] - visible['tier_median_ctr']

numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update', 'sessions_90d',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impressions_90d',
]
categorical_features = [
    'content_type', 'main_intent', 'competition_level',
    'freshness_tier', 'position_tier',
]

model_df = visible.copy()
for c in numeric_features:
    model_df[c] = model_df[c].fillna(0)
for c in categorical_features:
    model_df[c] = model_df[c].fillna('unknown')
model_df['log_impressions_90d'] = np.log1p(model_df['impressions_90d'])
numeric_features = [c if c != 'impressions_90d' else 'log_impressions_90d' for c in numeric_features]

X = model_df[numeric_features + categorical_features]
y = model_df['ctr_gap']
groups = model_df['client_id']
print(f'Reloaded {len(X):,} rows, same feature set as Week 5.')

Reloaded 12,023 rows, same feature set as Week 5.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I re-ran my exact Week-5 model twice: once under a **random row split** (rows shuffled and
split 80/20, ignoring which client each page belongs to — the leaky version), and once under
my actual **client-holdout split** from Week 5. Same features, same model, same random seed —
only the split changes.

In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score
import numpy as np

def run_split(train_idx, test_idx, label):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    test_df = model_df.iloc[test_idx]

    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)], remainder='passthrough')
    m = Pipeline([('pre', pre), ('rf', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))])
    m.fit(X_train, y_train)
    pred = m.predict(X_test)

    result = test_df.copy()
    result['predicted_ctr_gap'] = pred
    actual_top50 = set(result.sort_values('ctr_gap').head(50)['content_id'])
    model_top50 = set(result.sort_values('predicted_ctr_gap').head(50)['content_id'])
    p50 = len(actual_top50 & model_top50) / 50
    overlap = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
    r2 = r2_score(y_test, pred)
    print(f'{label}: R2={r2:.3f}  Precision@50={p50:.2f}  clients shared between train/test={overlap}')
    return r2, p50

# random row split (leaky -- same client's pages can land in both train and test)
idx = np.arange(len(X))
train_idx_r, test_idx_r = train_test_split(idx, test_size=0.2, random_state=42)
r2_leaky, p50_leaky = run_split(train_idx_r, test_idx_r, 'BEFORE (random row split)')

# client-holdout split (honest -- this is what w05 actually uses)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_g, test_idx_g = next(splitter.split(X, y, groups=groups))
r2_honest, p50_honest = run_split(train_idx_g, test_idx_g, 'AFTER  (client-holdout split)')

BEFORE (random row split): R2=0.426  Precision@50=0.10  clients shared between train/test=26
AFTER  (client-holdout split): R2=0.363  Precision@50=0.34  clients shared between train/test=0


**What changed.** R2 drops from the random split to the client-holdout split (0.43 → 0.36) —
that's the direction I expected: the random split lets the model see some of each client's
pages during training, so it partly memorizes client-specific quirks and looks a little better
than it really is on brand-new clients.

**What surprised me.** Precision@50 actually goes the other way (0.14 → 0.34, higher under the
honest split). I don't think this means the honest split is "leaking" in reverse — the two test
sets aren't the same size or the same client mix (the random split's test set is bigger and
spread across almost every client with only a few rows each; the grouped test set is a smaller
number of clients with more rows apiece), so ranking the true top 50 is a different, and
apparently easier, problem in the grouped case. I'm reporting both numbers honestly rather than
picking the one that favors my model. My takeaway: R2 is the cleaner leakage signal here, and
it confirms the client-holdout split is the more honest read on real generalization.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Re-checking my Week-5 feature list against the banned columns from the data dictionary:
`ctr` and `clicks_90d` (the pieces `ctr_gap` is built from), `trend_direction` and `trend_pct`
(the decline label's source), and `content_id` / `client_id` (identifiers, grouping only).

In [4]:
final_features = set(numeric_features) | set(categorical_features)
banned = {'ctr', 'clicks_90d', 'tier_median_ctr', 'trend_direction', 'trend_pct', 'content_id', 'client_id'}

touched = final_features & banned
print('Final feature set:', sorted(final_features))
print()
print('Banned columns touched by my final model:', touched if touched else 'none')
assert not touched, 'Leakage found -- a banned column made it into the feature list'
print('Leakage check passed.')

Final feature set: ['ai_traffic_pct', 'char_count', 'competition', 'competition_level', 'content_age_days', 'content_type', 'cpc', 'days_since_last_update', 'engagement_rate', 'freshness_tier', 'log_impressions_90d', 'main_intent', 'position_tier', 'scroll_rate', 'search_volume', 'sessions_90d', 'word_count']

Banned columns touched by my final model: none
Leakage check passed.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence, from Week 5:** "The model beats the baseline."

**Rewritten:** On this single 30,000-row teaching snapshot, under one client-holdout split, my
model's top-50 ranked pages overlapped with the true worst-`ctr_gap` pages about twice as often
as my Week-4 baseline rule's picks did (0.34 vs 0.16 Precision@50). This is an observed,
directional result on one split of one snapshot — it supports trying this approach on the full
warehouse release, not a guarantee that this exact margin holds at scale or across a different
time window.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.